# Monte Carlo Simulation — Regime-Switching Models

This notebook implements Monte Carlo methods for pricing financial derivatives under a two-regime volatility model. It covers:

- **Task 1**: Pricing a contract using standard and vectorised MC, convergence analysis, and variance reduction techniques  
- **Task 2**: Floating-strike Asian put pricing, convergence studies, and sensitivity analysis (dV/dλ)

## Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import timeit
from time import perf_counter
from scipy.stats import qmc, norm
from matplotlib.ticker import StrMethodFormatter
import matplotlib.cm as cm
import matplotlib.ticker as mticker
from time import perf_counter

---
## Task 1 — Parameters

Define the model parameters for the regime-switching contract: spot price, risk-free rate, maturity, strikes, volatilities, regime-switching rates, and the known exact value.

In [ ]:
S0, r, T   = 50085.5, 0.06, 2.0
X1, X2     = 50000, 55000
sigma_L, sigma_H = 0.22, 0.49
lam, mu    = 1.7, 2.2
exact_value = 34217.364

## Shared Helpers (Task 1)

Core functions used across Task 1:
- `payoff_from_st` — computes the contract payoff from terminal stock price  
- `simulate_tau_h_vectorised` — simulates time spent in the high-volatility regime  
- `discounted_payoff_from_tau_z` — converts (τ_H, z) into a discounted payoff

In [ ]:
def payoff_from_st(st):
    return np.where(st < X1, X1, np.where(st < X2, X2 - X1, st - X1))


def simulate_tau_h_vectorised(n_paths, rng):
    """Simulate time spent in high-vol regime for n_paths paths."""
    current_time  = np.zeros(n_paths)
    tau_high      = np.zeros(n_paths)
    in_high_vol   = np.zeros(n_paths, dtype=bool)   # all start in Low regime
    still_active  = np.ones(n_paths,  dtype=bool)

    while still_active.any():
        active_idx   = np.flatnonzero(still_active)
        time_left    = T - current_time[active_idx]
        switch_rates = np.where(in_high_vol[active_idx], mu, lam)
        wait_time    = rng.exponential(1.0 / switch_rates)
        step         = np.minimum(wait_time, time_left)

        tau_high[active_idx]     += step * in_high_vol[active_idx]
        current_time[active_idx] += step

        switched = wait_time < time_left
        in_high_vol[active_idx[switched]] = ~in_high_vol[active_idx[switched]]
        still_active[active_idx[current_time[active_idx] >= T]] = False

    return tau_high


def discounted_payoff_from_tau_z(tau_high, z):
    """Convert (tau_H, standard normal z) into discounted payoff."""
    sigma_hat_sq = (sigma_H**2 * tau_high + sigma_L**2 * (T - tau_high)) / T
    sigma_hat = np.sqrt(sigma_hat_sq)
    terminal_price = S0 * np.exp(
        (r - 0.5 * sigma_hat_sq) * T + sigma_hat * np.sqrt(T) * z
    )
    return np.exp(-r * T) * payoff_from_st(terminal_price)

---
## Task 1.1 — Monte Carlo Estimate with N = 100,000 (For Loop)

A path-by-path for-loop implementation of the Monte Carlo simulation. Each path independently simulates the regime-switching process and computes the discounted payoff.

In [ ]:
print("=" * 60)
print("TASK 1.1 — For-loop Monte Carlo, N = 100,000")
print("=" * 60)

def simulate_payoffs_loop(n_paths, seed=30):
    rng = np.random.default_rng(seed)
    payoffs = np.empty(n_paths)

    for i in range(n_paths):
        current_time = 0.0
        tau_high     = 0.0
        in_high_vol  = False          # start in Low regime

        while current_time < T:
            switch_rate = mu if in_high_vol else lam
            wait_time   = rng.exponential(1.0 / switch_rate)
            step        = min(wait_time, T - current_time)

            if in_high_vol:
                tau_high += step

            current_time += step
            if current_time < T:
                in_high_vol = not in_high_vol

        sigma_hat_sq = (sigma_H**2 * tau_high + sigma_L**2 * (T - tau_high)) / T
        z = rng.standard_normal()
        price_T = S0 * np.exp((r - 0.5 * sigma_hat_sq) * T + np.sqrt(sigma_hat_sq * T) * z)
        payoffs[i] = np.exp(-r * T) * payoff_from_st(price_T)

    return payoffs


N_task11 = 100_000
payoffs_loop = simulate_payoffs_loop(N_task11, seed=seed)
mean_loop    = payoffs_loop.mean()
se_loop      = payoffs_loop.std(ddof=1) / np.sqrt(N_task11)

print(f"N           = {N_task11:,}")
print(f"Estimate    = {mean_loop:.6f}")
print(f"Exact value = {exact_value:.6f}")
print(f"SE          = {se_loop:.6f}")
print(f"95% CI      = [{mean_loop - 1.96*se_loop:.6f}, {mean_loop + 1.96*se_loop:.6f}]")
print()

---
## Task 1.2 — Convergence Plot & Timing Comparison

### Vectorised MC with checkpoints

A batched vectorised implementation that records the running mean at user-specified checkpoints. Used for both convergence plots and timing benchmarks.

In [ ]:
print("=" * 60)
print("TASK 1.2 — Convergence plot & timing comparison")
print("=" * 60)

# --- Vectorised MC (batched, tracks running mean at checkpoints) ---
def mc_vectorised_checkpoints(n_total, checkpoints, batch_size=500_000, seed=30):
    """
    Run vectorised MC up to n_total paths, recording the running mean
    at each checkpoint.  Returns (final_mean, std_error, means_at_checkpoints, runtime).
    """
    rng         = np.random.default_rng(seed)
    checkpoints = np.array(sorted(set(int(x) for x in checkpoints if 1 <= x <= n_total)))
    means_at_cp = np.empty(len(checkpoints))

    running_sum = 0.0          # total payoff accumulated so far
    running_sum_sq = 0.0       # total squared payoff accumulated so far
    n_done = 0                 # number of simulated paths completed so far
    cp_idx = 0                 # index of the next checkpoint to fill
    tic = perf_counter()       # start time for runtime measurement

    while n_done < n_total:
        batch_n   = min(batch_size, n_total - n_done)
        tau_high  = simulate_tau_h_vectorised(batch_n, rng)
        z         = rng.standard_normal(batch_n)
        payoffs   = discounted_payoff_from_tau_z(tau_high, z)

        cum_sum   = np.cumsum(payoffs)
        batch_end = n_done + batch_n

        while cp_idx < len(checkpoints) and checkpoints[cp_idx] <= batch_end:
            k = checkpoints[cp_idx] - n_done - 1
            means_at_cp[cp_idx] = (running_sum + cum_sum[k]) / checkpoints[cp_idx]
            cp_idx += 1

        running_sum    += cum_sum[-1]
        running_sum_sq += np.cumsum(payoffs * payoffs)[-1]
        n_done          = batch_end

    final_mean = running_sum / n_total
    std_error  = np.sqrt(
        (running_sum_sq - running_sum**2 / n_total) / (n_total - 1) / n_total
    )
    return final_mean, std_error, means_at_cp, perf_counter() - tic

### Timing: For loop vs Vectorised at N = 1,000,000

In [ ]:
# --- Timing: for loop vs vectorised at N = 1,000,000 ---
N_timing = 1_000_000

time_loop = timeit.timeit(
    stmt="simulate_payoffs_loop(N_timing, seed)",
    globals=globals(),
    number=1
)
time_vec = timeit.timeit(
    stmt="mc_vectorised_checkpoints(N_timing, [N_timing], batch_size=500_000, seed=seed)",
    globals=globals(),
    number=1
)

print(f"Timing comparison at N = {N_timing:,}")
print(f"  For loop   : {time_loop:.2f} s")
print(f"  Vectorised : {time_vec:.2f} s")
print(f"  Speed-up   : {time_loop / time_vec:.1f}x")
print()

### Convergence Plot (N = 100,000)

Visualise how the MC estimate converges to the exact value as the number of paths increases, with a 95% confidence band.

In [ ]:
# --- Convergence plot (vectorised, N = 100,000) ---
N_conv = 100000
cp_conv = np.linspace(100, N_conv, 1000, dtype=int)
cp_conv[-1] = N_conv
mean_conv, se_conv, vals_conv, _ = mc_vectorised_checkpoints(
    N_conv, cp_conv, batch_size=500_000, seed=seed
)


se_cp = se_conv * np.sqrt(N_conv / cp_conv)
upper_bound = vals_conv + 1.96 * se_cp
lower_bound = vals_conv - 1.96 * se_cp

plt.figure(figsize=(10, 6))
plt.plot(cp_conv, vals_conv, color="blue", label="Mean")
plt.fill_between(
    cp_conv,
    lower_bound,
    upper_bound,
    color="gray",
    alpha=0.3,
    label="95% confidence band"
)
plt.axhline(exact_value, linestyle="--", color="red", label=f"Exact value = {exact_value}")
plt.xlabel("Number of paths (N)")
plt.ylabel("Contract value (V)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
print()

### Convergence Run (N = 20,000,000)

A large-scale convergence run to get a high-precision estimate.

In [ ]:
# --- Convergence run (N= 20,000,000) ---
N_conv = 20000000
cp_conv = np.linspace(100, N_conv, 1000, dtype=int)
cp_conv[-1] = N_conv
mean_conv, se_conv, vals_conv, _ = mc_vectorised_checkpoints(
    N_conv, cp_conv, batch_size=500_000, seed=seed
)

print(f"Convergence run (N = {N_conv:,})")
print(f"  Estimate    = {mean_conv:.6f}")
print(f"  Exact value = {exact_value:.6f}")
print(f"  SE          = {se_conv:.6f}")
print(f"  95% CI      = [{mean_conv - 1.96*se_conv:.6f}, {mean_conv + 1.96*se_conv:.6f}]")
print()

---
## Task 1.3 — Variance Reduction and Sampling Methods

### RNG Factory & Variance Reduction Utilities

Helper functions for different random number generators (PCG64, MT19937, Philox, SFC64, PCG64DXSM), quasi-random sequences (Halton, Sobol), and moment matching.

In [ ]:
# ============================================================
# TASK 1.3 — Variance reduction / sampling methods
# ============================================================
print("=" * 60)
print("TASK 1.3 — Variance reduction and sampling methods")
print("=" * 60)

# --- RNG factory ---
def make_rng(rng_name, seed):
    generators = {
        "PCG64"     : np.random.PCG64,
        "MT19937"   : np.random.MT19937,
        "Philox"    : np.random.Philox,
        "SFC64"     : np.random.SFC64,
        "PCG64DXSM" : np.random.PCG64DXSM
    }
    if rng_name not in generators:
        raise ValueError(f"Unknown RNG: {rng_name}")
    return np.random.Generator(generators[rng_name](seed))


# --- # Variance reduction & quasi-random utilities ---
def halton_to_normal(sampler, n):
    u = sampler.random(n).reshape(-1)               # ? sampler dùng bên dưới ntn?
    u = np.clip(u, 1e-12, 1 - 1e-12)
    return norm.ppf(u)

def sobol_to_normal(sampler, n):
    u = sampler.random(n).reshape(-1)
    u = np.clip(u, 1e-12, 1 - 1e-12)
    return norm.ppf(u)

def moment_match(z):
    sample_std = z.std(ddof=0)
    return (z - z.mean()) / sample_std if sample_std > 0 else z


# --- Single-batch simulator for a given method ---
def simulate_one_batch(n_paths, method, rng, halton_sampler=None, sobol_sampler=None):
    if method == "standard":
        tau_high = simulate_tau_h_vectorised(n_paths, rng)
        z        = rng.standard_normal(n_paths)
        return discounted_payoff_from_tau_z(tau_high, z)

    if method == "antithetic":
        half     = (n_paths + 1) // 2
        tau_high = simulate_tau_h_vectorised(half, rng)
        z        = rng.standard_normal(half)
        payoffs  = np.concatenate([
            discounted_payoff_from_tau_z(tau_high,  z),
            discounted_payoff_from_tau_z(tau_high, -z),
        ])
        return payoffs[:n_paths]

    if method == "moment_matching":
        tau_high = simulate_tau_h_vectorised(n_paths, rng)
        z        = moment_match(rng.standard_normal(n_paths))
        return discounted_payoff_from_tau_z(tau_high, z)

    if method == "halton":
        tau_high = simulate_tau_h_vectorised(n_paths, rng)
        z        = halton_to_normal(halton_sampler, n_paths)
        return discounted_payoff_from_tau_z(tau_high, z)

    if method == "sobol":
        tau_high = simulate_tau_h_vectorised(n_paths, rng)
        z        = sobol_to_normal(sobol_sampler, n_paths)
        return discounted_payoff_from_tau_z(tau_high, z)

    raise ValueError(f"Unknown method: {method}")


def run_single_seed(n_paths, method, seed, batch_size=100_000, rng_name="SFC64"):
    rng             = make_rng(rng_name, seed)
    halton_sampler  = qmc.Halton(d=1, scramble=True, seed=seed) if "halton" in method else None
    sobol_sampler  = qmc.Sobol(d=1, scramble=True, seed=seed)  if "sobol"  in method else None
    total_sum       = 0.0
    n_done          = 0

    while n_done < n_paths:
        batch_n    = min(batch_size, n_paths - n_done)
        total_sum += simulate_one_batch(batch_n, method, rng, halton_sampler, sobol_sampler).sum()
        n_done    += batch_n

    estimate   = total_sum / n_paths
    abs_pct_err = abs(estimate - exact_value) / exact_value * 100
    return estimate, abs_pct_err


def run_experiment(method_or_rng, n_paths_per_seed, n_seeds, batch_size,
                   base_seed, rng_name="SFC64", is_rng_test=False):
    estimates   = np.empty(n_seeds)
    abs_pct_err = np.empty(n_seeds)
    method      = "standard" if is_rng_test else method_or_rng

    tic = perf_counter()
    for i in range(n_seeds):
        rng_used = method_or_rng if is_rng_test else rng_name
        est, err = run_single_seed(n_paths_per_seed, method, base_seed + i,
                                   batch_size, rng_used)
        estimates[i]   = est
        abs_pct_err[i] = err
    elapsed = perf_counter() - tic

    value_est    = estimates.mean()
    avg_abs_err  = abs_pct_err.mean()
    overall_err  = abs(value_est - exact_value) / exact_value * 100

    return {
        "label"      : method_or_rng,
        "value_est"  : value_est,
        "overall_err": overall_err,
        "avg_abs_err": avg_abs_err,
        "runtime"    : elapsed,
        "all_estimates": estimates,
    }

### 1.3-A: Compare RNG Generators

Run each RNG with 100 seeds × 1,000,000 paths to compare runtime and accuracy.

In [ ]:
# --- 1.3-A: Compare RNG generators ---
rng_names          = ["PCG64", "MT19937", "Philox", "SFC64", "PCG64DXSM"]
paths_per_seed_rng = 1000000
n_seeds_rng        = 100
batch_rng          = 500_000

print(f"RNG comparison: {n_seeds_rng} seeds x {paths_per_seed_rng:,} paths each")
print(f"{'RNG':8s}  {'Estimate':>12s}  {'Abs%Err':>10s}  {'AvgAbs%Err':>12s}  {'Time(s)':>9s}")

results_rng = []
for rng_name in rng_names:
    res = run_experiment(rng_name, paths_per_seed_rng, n_seeds_rng,
                         batch_rng, base_seed=30, is_rng_test=True)
    results_rng.append(res)
    print(f"{res['label']:8s}  {res['value_est']:12.6f}  "
          f"{res['overall_err']:10.6f}%  {res['avg_abs_err']:12.6f}%  "
          f"{res['runtime']:9.2f}")
print()

### Plot: RNG Comparison (Runtime & Accuracy)

In [ ]:
# --- Plotting the figure to compare different generators ---
labels   = [r["label"] for r in results_rng]
runtime  = [r["runtime"] for r in results_rng]
accuracy = [r["avg_abs_err"] for r in results_rng]

x = np.arange(len(labels))
w = 0.28
g = 0.08

fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()

ax1.bar(x - w/2 - g/2, runtime,  w, color="tab:blue",   label="Runtime (s)")
ax2.bar(x + w/2 + g/2, accuracy, w, color="tab:orange", label="Avg abs % error")

ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel("Runtime (seconds)")
ax2.set_ylabel("Average Absolute % Error")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()

ax1.legend(
    h1 + h2, l1 + l2,
    loc="upper right",
    bbox_to_anchor=(1, 1.1),
    borderaxespad=0.0
)

ax1.grid(axis="y")
fig.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()

### 1.3-B: Compare Variance-Reduction Methods

Compare Standard MC, Antithetic Variates, Moment Matching, Halton, and Sobol across 100 seeds × 1,000,000 paths.

In [ ]:
# --- 1.3-B: Compare variance-reduction methods ---
method_names       = ["standard", "antithetic", "moment_matching",
                      "halton", "sobol"]
paths_per_seed_vr  = 1_000_000
n_seeds_vr         = 100
batch_vr           = 100_000

print(f"Variance-reduction comparison: {n_seeds_vr} seeds x {paths_per_seed_vr:,} paths each")
print(f"{'Method':28s}  {'Estimate':>12s}  {'Abs%Err':>10s}  {'AvgAbs%Err':>12s}  {'Time(s)':>9s}")

results_method = []
for method in method_names:
    res = run_experiment(method, paths_per_seed_vr, n_seeds_vr,
                         batch_vr, base_seed=30, rng_name="SFC64")
    results_method.append(res)
    print(f"{res['label']:28s}  {res['value_est']:12.6f}  "
          f"{res['overall_err']:10.6f}%  {res['avg_abs_err']:12.6f}%  "
          f"{res['runtime']:9.2f}")

# --- Plotting the figure to compare different methods ---
labels_raw = [r["label"] for r in results_method]
runtime    = [r["runtime"] for r in results_method]
accuracy   = [r["avg_abs_err"] for r in results_method]

labels = [s.replace("_"," ").title() for s in labels_raw]

x = np.arange(len(labels))
w = 0.28
gap = 0.08
fig, ax1 = plt.subplots(figsize=(10,6))
ax2 = ax1.twinx()
ax1.bar(x - w/2 - gap/2, runtime,  w, color="tab:blue",   label="Runtime (s)")
ax2.bar(x + w/2 + gap/2, accuracy, w, color="tab:orange", label="Avg Abs % Error")
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel("Runtime (seconds)")
ax2.set_ylabel("Average Absolute % Error")
h1,l1 = ax1.get_legend_handles_labels()
h2,l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc="upper right")
ax1.grid(axis="y")
plt.tight_layout()
plt.show()

### 1.3-C: Percentage Error & Runtime vs N (10K → 1M)

Sweep the number of paths from 10,000 to 1,000,000 and track both percentage error and runtime for each method.

In [ ]:
# ============================================================
# TASK 1.3-C — % Error vs N  &  Runtime vs N  (Figure 4 & 5)
 
print("=" * 60)
print("TASK 1.3-D — % Error & Runtime vs N  (10K → 1M)")
print("=" * 60)
 
N_sweep_13d = list(range(10_000, 1_001_000, 50_000))   # 20 points
 
methods_13d     = ["standard", "antithetic", "moment_matching", "halton", "sobol"]
labels_13d      = ["Standard MC", "Antithetic Variates",
                   "Moment Matching", "Halton", "Sobol"]
colors_13d      = ["tab:blue", "tab:orange", "green", "black", "tab:purple"]
markers_13d     = ["o", "s", "^", "D", "v"]
 
pct_err_13d = {m: [] for m in methods_13d}
runtime_13d = {m: [] for m in methods_13d}
 
print(f"{'N':>10}  " + "  ".join(f"{m[:8]:>10}" for m in methods_13d))
print("-" * 75)
 
for N in N_sweep_13d:
    row_err = []
    for method in methods_13d:
        # ── single run for the % error estimate ──────────────────────
        rng_m = make_rng("SFC64", seed)
        hs    = qmc.Halton(d=1, scramble=True, seed=seed) if method == "halton" else None
        ss    = qmc.Sobol( d=1, scramble=True, seed=seed) if method == "sobol"  else None
        payoffs = simulate_one_batch(N, method, rng_m, hs, ss)
        pct_err = (payoffs.mean() - exact_value) / exact_value * 100
        pct_err_13d[method].append(pct_err)
        row_err.append(pct_err)
 
        # ── timeit: GC is disabled automatically → eliminates GC spikes ─
        # repeat=1 is sufficient: each run already takes hundreds of ms,
        # so OS jitter is negligible and repeating 3× just wastes time.
        elapsed = timeit.timeit(
            lambda m=method: simulate_one_batch(
                N, m,
                make_rng("SFC64", seed),
                qmc.Halton(d=1, scramble=True, seed=seed) if m == "halton" else None,
                qmc.Sobol( d=1, scramble=True, seed=seed) if m == "sobol"  else None,
            ),
            number=1,
        )
        runtime_13d[method].append(elapsed)
 
    print(f"{N:>10,}  " + "  ".join(f"{v:+10.4f}" for v in row_err))
 
print()
 
# ── Figure 4: % Error vs N ────────────────────────────────────────────
fig4, ax4 = plt.subplots(figsize=(10, 6))
 
for method, label, color, marker in zip(
        methods_13d, labels_13d, colors_13d, markers_13d):
    ax4.plot(N_sweep_13d, pct_err_13d[method],
             color=color, marker=marker, linestyle="-",
             linewidth=1.6, markersize=5, label=label)
 
ax4.axhline(0.0, color="red", linestyle="--", linewidth=1.2,
            label="Exact value (0% error)")
ax4.set_xlabel("Number of Paths ($N$)", fontsize=12)
ax4.set_ylabel(r"Percentage Standard Error", fontsize=12)
ax4.legend(loc="upper right", fontsize=10)
ax4.grid(True, alpha=0.4)
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()
 
# ── Figure 5: Runtime vs N ────────────────────────────────────────────
fig5, ax5 = plt.subplots(figsize=(10, 6))
 
for method, label, color, marker in zip(
        methods_13d, labels_13d, colors_13d, markers_13d):
    ax5.plot(N_sweep_13d, runtime_13d[method], color=color, linestyle="-",
             linewidth=1.9, marker=marker, markersize=5, label=label)
 
ax5.set_xlabel("Number of Paths ($N$)", fontsize=12)
ax5.set_ylabel("Runtime (seconds)", fontsize=12)
ax5.set_title("Figure 5: Runtime vs $N$ for 5 Simulation Methods  (Task 1.3)", fontsize=13)
ax5.legend(loc="upper left", fontsize=10)
ax5.grid(True, alpha=0.4)
ax5.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig("fig_task13d_runtime_vs_N.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved  →  fig_task13d_runtime_vs_N.png")
print()

---
## Task 2 — Floating-Strike Asian Put

### Parameters

In [ ]:
# ============================================================
S0_2    = 55.0209
r_2     = 0.04
T_2     = 1.5
K_obs   = 60
sigma_L2 = 0.18
sigma_H2 = 0.34
lam_2   = 2.0
mu_2    = 3.0

### Core Path Simulators

Multiple simulation engines for the floating-strike Asian put, each implementing a different variance-reduction strategy:

- `simulate_asian_put` — standard vectorised MC  
- `simulate_asian_put_antithetic` — antithetic variates  
- `simulate_asian_put_moment_matching` — moment-matched normals  
- `simulate_asian_put_halton` — quasi-MC with scrambled Halton  
- `simulate_asian_put_sobol` — quasi-MC with scrambled Sobol  
- `simulate_asian_put_crn` — common random numbers for sensitivity analysis  
- `mean_se` — helper to compute mean and standard error

In [ ]:
# Core path simulator (used by all Task 2 sections)
# ============================================================
def simulate_asian_put(n_paths, K=K_obs, lam_val=lam_2, seed=30):
    """
    Vectorised Monte Carlo for floating-strike Asian put.
    Payoff: max(A - S_T, 0) where A = arithmetic mean of S(t_1)..S(t_K).
    lam_val is exposed so Task 2.3 can bump it without touching globals.
    """
    rng      = np.random.default_rng(seed)
    dt       = T_2 / K
    prob_L_H = lam_val * dt    # probability of switching L -> H in one step
    prob_H_L = mu_2     * dt    # probability of switching H -> L in one step

    price     = np.full(n_paths, S0_2, dtype=np.float64)
    in_high   = np.zeros(n_paths, dtype=bool)   # all start in Low
    sum_price = np.zeros(n_paths)
    for _ in range(K):
        u_switch = rng.random(n_paths)              # probability of switching the regime
        switched = np.where(in_high, u_switch < prob_H_L, u_switch < prob_L_H)
        in_high  = np.where(switched, ~in_high, in_high)
        vol   = np.where(in_high, sigma_H2, sigma_L2)
        z     = rng.standard_normal(n_paths)
        price = price * np.exp((r_2 - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * z)
        sum_price += price

    avg_price = sum_price / K
    payoff    = np.maximum(avg_price - price, 0.0)
    return np.exp(-r_2 * T_2) * payoff


def simulate_asian_put_antithetic(n_paths, K=K_obs, lam_val=lam_2, seed=30):
    """
    Antithetic variates: pair each path with its sign-flipped normal draws.
    Switching uniforms are shared between original and antithetic paths.
    Returns n_paths discounted payoffs (using ceil(n_paths/2) base paths).
    """
    half     = (n_paths + 1) // 2
    rng      = np.random.default_rng(seed)
    dt       = T_2 / K
    prob_L_H = lam_val * dt
    prob_H_L = mu_2     * dt

    price_orig  = np.full(half, S0_2, dtype=np.float64)
    price_anti  = np.full(half, S0_2, dtype=np.float64)
    in_high_orig = np.zeros(half, dtype=bool)
    in_high_anti = np.zeros(half, dtype=bool)
    sum_orig     = np.zeros(half)
    sum_anti     = np.zeros(half)

    for _ in range(K):
        u_switch     = rng.random(half)
        switched_orig = np.where(in_high_orig, u_switch < prob_H_L, u_switch < prob_L_H)
        switched_anti = np.where(in_high_anti, u_switch < prob_H_L, u_switch < prob_L_H)
        in_high_orig  = np.where(switched_orig, ~in_high_orig, in_high_orig)
        in_high_anti  = np.where(switched_anti, ~in_high_anti, in_high_anti)

        vol_orig = np.where(in_high_orig, sigma_H2, sigma_L2)
        vol_anti = np.where(in_high_anti, sigma_H2, sigma_L2)

        z            = rng.standard_normal(half)
        price_orig   = price_orig * np.exp((r_2 - 0.5*vol_orig**2)*dt + vol_orig*np.sqrt(dt)* z)
        price_anti   = price_anti * np.exp((r_2 - 0.5*vol_anti**2)*dt + vol_anti*np.sqrt(dt)*-z)
        sum_orig    += price_orig
        sum_anti    += price_anti

    disc     = np.exp(-r_2 * T_2)
    pay_orig = disc * np.maximum(sum_orig / K - price_orig, 0.0)
    pay_anti = disc * np.maximum(sum_anti / K - price_anti, 0.0)
    return np.concatenate([pay_orig, pay_anti])[:n_paths]


def simulate_asian_put_moment_matching(n_paths, K=K_obs, lam_val=lam_2, seed=30):
    """
    Moment-matched MC: at every time step, force the batch of N normals
    to have sample mean = 0 and sample std = 1 before computing returns.
    """
    rng      = np.random.default_rng(seed)
    dt       = T_2 / K
    prob_L_H = lam_val * dt
    prob_H_L = mu_2     * dt

    price     = np.full(n_paths, S0_2, dtype=np.float64)
    in_high   = np.zeros(n_paths, dtype=bool)
    sum_price = np.zeros(n_paths)

    for _ in range(K):
        u_switch = rng.random(n_paths)
        switched = np.where(in_high, u_switch < prob_H_L, u_switch < prob_L_H)
        in_high  = np.where(switched, ~in_high, in_high)
        vol      = np.where(in_high, sigma_H2, sigma_L2)
        z_raw    = rng.standard_normal(n_paths)
        s        = z_raw.std(ddof=0)
        z        = (z_raw - z_raw.mean()) / s if s > 0 else z_raw   # match mean & std
        price    = price * np.exp((r_2 - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * z)
        sum_price += price

    avg_price = sum_price / K
    payoff    = np.maximum(avg_price - price, 0.0)
    return np.exp(-r_2 * T_2) * payoff


def simulate_asian_put_halton(n_paths, K=K_obs, lam_val=lam_2, seed=30):
    """
    Quasi-MC with K-dimensional scrambled Halton sequence.
    Each path dimension corresponds to one time step's normal draw.
    Regime-switch uniforms remain pseudo-random.
    """
    rng      = np.random.default_rng(seed)
    dt       = T_2 / K
    prob_L_H = lam_val * dt
    prob_H_L = mu_2     * dt

    halton = qmc.Halton(d=K, scramble=True, seed=seed)
    u      = halton.random(n_paths)                    # (n_paths, K)
    u      = np.clip(u, 1e-12, 1 - 1e-12)
    z_all  = norm.ppf(u)                               # (n_paths, K)

    price     = np.full(n_paths, S0_2, dtype=np.float64)
    in_high   = np.zeros(n_paths, dtype=bool)
    sum_price = np.zeros(n_paths)

    for k in range(K):
        u_switch = rng.random(n_paths)
        switched = np.where(in_high, u_switch < prob_H_L, u_switch < prob_L_H)
        in_high  = np.where(switched, ~in_high, in_high)
        vol      = np.where(in_high, sigma_H2, sigma_L2)
        z        = z_all[:, k]
        price    = price * np.exp((r_2 - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * z)
        sum_price += price

    avg_price = sum_price / K
    payoff    = np.maximum(avg_price - price, 0.0)
    return np.exp(-r_2 * T_2) * payoff


def simulate_asian_put_sobol(n_paths, K=K_obs, lam_val=lam_2, seed=30):
    """
    Quasi-MC with K-dimensional scrambled Sobol sequence.
    Rounds up n_paths to next power of 2 for optimal Sobol properties,
    then truncates to n_paths.
    """
    rng      = np.random.default_rng(seed)
    dt       = T_2 / K
    prob_L_H = lam_val * dt
    prob_H_L = mu_2     * dt

    sobol    = qmc.Sobol(d=K, scramble=True, seed=seed)
    m        = max(1, int(np.ceil(np.log2(max(n_paths, 2)))))
    n_sobol  = 2 ** m
    u        = sobol.random(n_sobol)[:n_paths]         # (n_paths, K)
    u        = np.clip(u, 1e-12, 1 - 1e-12)
    z_all    = norm.ppf(u)                             # (n_paths, K)

    price     = np.full(n_paths, S0_2, dtype=np.float64)
    in_high   = np.zeros(n_paths, dtype=bool)
    sum_price = np.zeros(n_paths)

    for k in range(K):
        u_switch = rng.random(n_paths)
        switched = np.where(in_high, u_switch < prob_H_L, u_switch < prob_L_H)
        in_high  = np.where(switched, ~in_high, in_high)
        vol      = np.where(in_high, sigma_H2, sigma_L2)
        z        = z_all[:, k]
        price    = price * np.exp((r_2 - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * z)
        sum_price += price

    avg_price = sum_price / K
    payoff    = np.maximum(avg_price - price, 0.0)
    return np.exp(-r_2 * T_2) * payoff


def simulate_asian_put_crn(n_paths, K=K_obs, lam_val=lam_2, delta_lam=0.1, seed=30):
    """
    Common Random Numbers: run base (lambda) and bumped (lambda + delta_lam)
    paths from the same random seed so that paths are correlated.
    Returns (payoffs_base, payoffs_bumped).
    """
    rng      = np.random.default_rng(seed)
    dt       = T_2 / K
    prob_L_H_base   = lam_val             * dt
    prob_L_H_bumped = (lam_val + delta_lam) * dt
    prob_H_L        = mu_2 * dt

    price_base   = np.full(n_paths, S0_2, dtype=np.float64)
    price_bumped = np.full(n_paths, S0_2, dtype=np.float64)
    in_high_base   = np.zeros(n_paths, dtype=bool)
    in_high_bumped = np.zeros(n_paths, dtype=bool)
    sum_base   = np.zeros(n_paths)
    sum_bumped = np.zeros(n_paths)

    for _ in range(K):
        u_switch       = rng.random(n_paths)
        switched_base  = np.where(in_high_base,   u_switch < prob_H_L, u_switch < prob_L_H_base)
        switched_bumped = np.where(in_high_bumped, u_switch < prob_H_L, u_switch < prob_L_H_bumped)
        in_high_base   = np.where(switched_base,   ~in_high_base,   in_high_base)
        in_high_bumped = np.where(switched_bumped, ~in_high_bumped, in_high_bumped)

        vol_base   = np.where(in_high_base,   sigma_H2, sigma_L2)
        vol_bumped = np.where(in_high_bumped, sigma_H2, sigma_L2)

        z            = rng.standard_normal(n_paths)
        price_base   = price_base   * np.exp((r_2 - 0.5*vol_base**2)  *dt + vol_base  *np.sqrt(dt)*z)
        price_bumped = price_bumped * np.exp((r_2 - 0.5*vol_bumped**2)*dt + vol_bumped*np.sqrt(dt)*z)
        sum_base   += price_base
        sum_bumped += price_bumped

    disc        = np.exp(-r_2 * T_2)
    pay_base    = disc * np.maximum(sum_base   / K - price_base,   0.0)
    pay_bumped  = disc * np.maximum(sum_bumped / K - price_bumped, 0.0)
    return pay_base, pay_bumped


def mean_se(payoffs):
    n = len(payoffs)
    return payoffs.mean(), payoffs.std(ddof=1) / np.sqrt(n)

---
## Task 2.1 — Asian Put Estimate (N = 1,000,000) & Convergence Plot

Compute the option value and visualise how the running mean converges with a 95% confidence band.

In [ ]:
# ============================================================
# TASK 2.1 — Asian put estimate with N = 1,000,000
#            + convergence plot with 95% confidence band
# ============================================================
print("=" * 60)
print("TASK 2.1 — Floating-Strike Asian Put, N = 1,000,000")
print("=" * 60)

N_task21    = 1_000_000
payoffs_21  = simulate_asian_put(N_task21, K=K_obs, seed=30)
mean_21, se_21 = mean_se(payoffs_21)

print(f"Parameters: S0={S0_2}, r={r_2}, T={T_2}, K={K_obs}, "
      f"sigma_L2={sigma_L2}, sigma_H2={sigma_H2}, lam_2={lam_2}, mu_2={mu_2}")
print(f"N           = {N_task21:,}")
print(f"Estimate    = {mean_21:.6f}")
print(f"SE          = {se_21:.6f}")
print(f"95% CI      = [{mean_21 - 1.96*se_21:.6f}, {mean_21 + 1.96*se_21:.6f}]")
print(f"(Known approximate value: ~3)")
print()

# --- Build checkpoints from already-simulated payoffs (no re-simulation needed) ---
cp_21 = np.unique(np.concatenate([
    np.arange(1_000,  10_001,  1_000),
    np.arange(10_000, 100_001, 5_000),
    np.arange(100_000, 1_000_001, 50_000),
])).astype(int)
cp_21 = cp_21[cp_21 <= N_task21]
cp_21[-1] = N_task21

# Running statistics via cumulative sums  (O(N) memory, single pass)
cum_sum_21    = np.cumsum(payoffs_21)
cum_sum_sq_21 = np.cumsum(payoffs_21 ** 2)

running_means_21 = cum_sum_21[cp_21 - 1] / cp_21
# Unbiased running variance via Welford-equivalent formula
running_vars_21  = (
    cum_sum_sq_21[cp_21 - 1] / cp_21 - running_means_21 ** 2
) * (cp_21 / (cp_21 - 1))
running_se_21    = np.sqrt(np.maximum(running_vars_21, 0.0) / cp_21)

upper_21 = running_means_21 + 1.96 * running_se_21
lower_21 = running_means_21 - 1.96 * running_se_21


fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(cp_21, running_means_21, color="blue", linewidth=1.5, label="Mean")
ax.fill_between(cp_21, lower_21, upper_21, color="lightgray", alpha=0.8, label="95% confidence band")
ax.axhline(mean_21, color="red", linestyle="--", linewidth=1.3, label=f"Exact value = {mean_21:.3f}")
ax.set_xlabel("Number of paths (N)")
ax.set_ylabel("Option value (V)")
ax.set_xticks(np.arange(0, cp_21[-1] + 1, 200_000))
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.legend(loc="upper right")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
print()

---
## Task 2.2 — Convergence Study (SFC64 + Antithetic Variates)

### Simulator: SFC64 + Antithetic Variates (Vectorised, Batched)

A production-grade simulator using the SFC64 generator with antithetic variates and configurable batch sizes.

In [ ]:
# -*- coding: utf-8 -*-
"""
TASK 2.2 — Convergence study for floating-strike Asian put
Method : NumPy vectorisation + RNG SFC64 + antithetic variates
Output : one summary table  +  one figure (V vs N, coloured by K)
"""

# ── Core simulator: SFC64 + antithetic variates (vectorised) ──────────────────
def simulate_asian_put_sfc64_anti(n_paths, K,
                                   S0=S0_2, r=r_2, T=T_2,
                                   sig_L=sigma_L2, sig_H=sigma_H2,
                                   lam=lam_2, mu=mu_2, seed=seed,
                                   batch_size=500_000):
    """
    Floating-strike Asian put via SFC64 + antithetic variates.

    Antithetic construction
    -----------------------
    * Generate half = ceil(N/2) paths using normal draws z.
    * Pair each path with its mirror using -z (same regime-switch uniforms,
      so paths share identical volatility trajectories).
    * Average over all N = 2*half (truncated to n_paths) discounted payoffs.

    Batching
    --------
    Large n_paths are split into batches of `batch_size` to keep per-batch
    memory below ~50 MB regardless of K.
    """
    rng      = np.random.Generator(np.random.SFC64(seed))
    dt       = T / K
    sqrt_dt  = np.sqrt(dt)
    prob_L_H = lam * dt          # P(L→H in one Euler step)
    prob_H_L = mu  * dt          # P(H→L in one Euler step)
    disc     = np.exp(-r * T)

    all_payoffs = []
    n_done      = 0

    while n_done < n_paths:
        # ── batch size (n_paths may not divide evenly) ────────────────────
        batch_full = min(batch_size, n_paths - n_done)   # full paths this batch
        half       = (batch_full + 1) // 2               # antithetic base paths

        # initialise prices and regime state
        price_o  = np.full(half, S0,  dtype=np.float64)
        price_a  = np.full(half, S0,  dtype=np.float64)
        in_hi_o  = np.zeros(half, dtype=bool)   # all start in Low regime
        in_hi_a  = np.zeros(half, dtype=bool)
        sum_o    = np.zeros(half)
        sum_a    = np.zeros(half)

        # ── step through K time intervals ─────────────────────────────────
        for _ in range(K):
            # Shared switching uniform → both paths experience the same
            # regime draw, preserving the variance-reduction benefit.
            u       = rng.random(half)
            sw_o    = np.where(in_hi_o, u < prob_H_L, u < prob_L_H)
            sw_a    = np.where(in_hi_a, u < prob_H_L, u < prob_L_H)
            in_hi_o = np.where(sw_o, ~in_hi_o, in_hi_o)
            in_hi_a = np.where(sw_a, ~in_hi_a, in_hi_a)

            vol_o   = np.where(in_hi_o, sig_H, sig_L)
            vol_a   = np.where(in_hi_a, sig_H, sig_L)

            z       = rng.standard_normal(half)          # SFC64 draw
            price_o = price_o * np.exp((r - 0.5*vol_o**2)*dt + vol_o*sqrt_dt* z)
            price_a = price_a * np.exp((r - 0.5*vol_a**2)*dt + vol_a*sqrt_dt*-z)
            sum_o  += price_o
            sum_a  += price_a

        # ── payoffs for this batch ─────────────────────────────────────────
        pay_o = disc * np.maximum(sum_o/K - price_o, 0.0)
        pay_a = disc * np.maximum(sum_a/K - price_a, 0.0)
        batch_payoffs = np.concatenate([pay_o, pay_a])[:batch_full]

        all_payoffs.append(batch_payoffs)
        n_done += batch_full

    return np.concatenate(all_payoffs)

### Run All (N, K) Combinations

Grid sweep over 6 values of N (100K–5M) and 7 values of K (5–252 time steps).

In [ ]:
# ── Grid: deliberately log-spaced in N, targeted K values ─────────────────────
# N: 6 points, 100 K → 5 M.  Steps are wider at large N to save compute
#    while still resolving the 1/√N convergence curve.
# K: 7 points from coarse (5) to near-daily (252).  K=60 is the target.
N_values = [100_000, 250_000, 500_000, 1_000_000, 2_000_000, 5_000_000]
K_values = [5, 10, 20, 60, 90, 126, 252]

# ── Run all (N, K) combinations ───────────────────────────────────────────────
print("=" * 70)
print("TASK 2.2  —  SFC64 + Antithetic Variates  convergence grid")
print("=" * 70)
print(f"{'#':>3}  {'K':>4}  {'N':>9}  {'V̂':>9}  {'SE':>8}  {'95% CI':>22}  {'time':>6}")
print("-" * 70)

results = {}       # (N, K) -> (mean, se)
total   = len(N_values) * len(K_values)
done    = 0
t_all   = perf_counter()

for K in K_values:
    for N in N_values:
        tic = perf_counter()
        pay = simulate_asian_put_sfc64_anti(N, K=K, seed=seed)
        m   = pay.mean()
        se  = pay.std(ddof=1) / np.sqrt(N)
        results[(N, K)] = (m, se)
        done += 1
        elapsed = perf_counter() - tic
        ci_lo, ci_hi = m - 1.96*se, m + 1.96*se
        print(f"{done:3d}  K={K:>3d}  N={N:>9,}  "
              f"{m:9.5f}  {se:8.6f}  [{ci_lo:.5f}, {ci_hi:.5f}]  {elapsed:5.1f}s")

print(f"\nTotal wall-clock time: {perf_counter()-t_all:.1f} s")

### Best Estimate (K = 252, N = 5,000,000)

In [ ]:
# ============================================================
# TASK 2.2 — Best estimate: K = 252, N = 5,000,000
# ============================================================
print("=" * 60)
print("TASK 2.2 — Best estimate  (K = 252, N = 5,000,000)")
print("=" * 60)
 
K_best = 252
N_best = 5_000_000
 
tic_best = perf_counter()
payoffs_best = simulate_asian_put_sfc64_anti(N_best, K=K_best, seed=seed)
elapsed_best = perf_counter() - tic_best
 
mean_best = payoffs_best.mean()
se_best   = payoffs_best.std(ddof=1) / np.sqrt(N_best)
ci_lo_best = mean_best - 1.96 * se_best
ci_hi_best = mean_best + 1.96 * se_best
 
print(f"  Method    : SFC64 + Antithetic Variates (vectorised)")
print(f"  K         : {K_best}  (≈ daily observations over T = {T_2} yr)")
print(f"  N         : {N_best:,}")
print(f"  Estimate  : {mean_best:.6f}")
print(f"  Std Error : {se_best:.6f}")
print(f"  95%% CI   : [{ci_lo_best:.6f}, {ci_hi_best:.6f}]")
print(f"  Wall time : {elapsed_best:.2f} s")
print()
print(f"  >>> Best estimate of V (K=252, N=5M): {mean_best:.4f} ± {1.96*se_best:.4f} (95% CI)")

### Figures: Line Chart (V̂ vs N) & Heatmap (N × K)

In [ ]:
# ============================================================
# TASK 2.2 — Figures 
# ============================================================
 
from matplotlib.colors import LinearSegmentedColormap
 
# ── Simulate additional small-N points so line chart starts at 10K ─
N_extra = [10_000, 25_000, 50_000]
print("Simulating small-N points for line chart...")
for K in K_values:
    for N in N_extra:
        pay = simulate_asian_put_sfc64_anti(N, K=K, seed=seed)
        results[(N, K)] = (pay.mean(), pay.std(ddof=1) / np.sqrt(N))
 
# ── Ordered N lists ────────────────────────────────────────────────
N_all  = sorted({k[0] for k in results})          # all N → heatmap
N_plot = [N for N in N_all if N <= 1_000_000]     # 10K–1M → line chart
N_arr  = np.array(N_plot, dtype=float)
 
# ══════════════════════════════════════════════════════════════════
# Figure 1 — Line chart  (V̂ vs N, 10K–1M, coloured by K)
# ══════════════════════════════════════════════════════════════════
line_colours = [plt.colormaps["tab10"](i) for i in range(len(K_values))]
 
fig, ax = plt.subplots(figsize=(10, 6))
for idx, K in enumerate(K_values):
    ms = np.array([results[(N, K)][0] for N in N_plot])
    ax.plot(N_arr, ms, "o-", color=line_colours[idx], linewidth=1.6,
            markersize=5, label=f"K = {K}")
 
ax.set_xscale("log")
ax.set_xlim(N_plot[0] * 0.85, N_plot[-1] * 1.15)
 
# Major ticks: one labelled tick per round value
_major_ticks  = [10_000, 25_000, 50_000,
                 100_000, 250_000, 500_000, 1_000_000]
_major_labels = ["10K", "25K", "50K",
                 "100K", "250K", "500K", "1M"]
ax.set_xticks(_major_ticks)
ax.set_xticklabels(_major_labels, fontsize=9)
 
# Minor ticks: dense log-spaced lines between decades
ax.xaxis.set_minor_locator(
    mticker.LogLocator(base=10, subs=[2, 3, 4, 5, 6, 7, 8, 9], numticks=20))
ax.xaxis.set_minor_formatter(mticker.NullFormatter())
 
ax.grid(True, which="major", alpha=0.50, linewidth=0.9)
ax.grid(True, which="minor", alpha=0.25, linewidth=0.5, linestyle=":")
ax.set_xlabel("Number of paths ($N$)", fontsize=12)
ax.set_ylabel(r"Option value ($\hat{V}$)", fontsize=12)
ax.legend(fontsize=10, loc="lower right", ncol=2, title="Time steps ($K$)")
plt.tight_layout()
plt.show()
 
# ══════════════════════════════════════════════════════════════════
# Figure 2 — Heatmap  (all N × K, green → yellow → orange → red)
# ══════════════════════════════════════════════════════════════════
_gyr = LinearSegmentedColormap.from_list(
    "gyr",
    [(0.00, "#3a9e3a"),   # dark green  (lowest V̂)
     (0.30, "#7dc73d"),   # light green
     (0.50, "#f5d800"),   # yellow
     (0.70, "#f0900a"),   # orange
     (0.85, "#e03000"),   # orange-red
     (1.00, "#cc0000")])  # deep red    (highest V̂)
 
heatmap_data  = np.array([[results[(N, K)][0] for N in N_all] for K in K_values])
col_labels_hm = [f"{N // 1_000}K" for N in N_all]
row_labels_hm = [str(K) for K in K_values]
 
fig2, ax2 = plt.subplots(figsize=(14, 4.5))
im   = ax2.imshow(heatmap_data, aspect="auto", cmap=_gyr)
cbar = fig2.colorbar(im, ax=ax2, pad=0.02, fraction=0.03)
cbar.set_label(r"Estimated option value ($\hat{V}$)", fontsize=11)
 
ax2.set_xticks(range(len(N_all)))
ax2.set_xticklabels(col_labels_hm, fontsize=9)
ax2.set_yticks(range(len(K_values)))
ax2.set_yticklabels(row_labels_hm, fontsize=10)
ax2.set_xlabel("Number of paths ($N$)", fontsize=11)
ax2.set_ylabel("Time steps ($K$)", fontsize=11)
 
_vmin, _vmax = heatmap_data.min(), heatmap_data.max()
for i in range(len(K_values)):
    for j in range(len(N_all)):
        val        = heatmap_data[i, j]
        brightness = (val - _vmin) / (_vmax - _vmin)
        txt_colour = "white" if brightness < 0.20 or brightness > 0.80 else "black"
        ax2.text(j, i, f"{val:.4f}", ha="center", va="center",
                 fontsize=7.5, color=txt_colour, fontweight="bold")
 
plt.tight_layout()
plt.show()

---
## Task 2.3 — Estimating dV/dλ (Sensitivity to Regime-Switching Rate)

### Hyper-parameters & CRN Derivative Engine

The CRN engine simulates paired paths under λ and λ+Δλ sharing the same random stream, exploiting covariance for lower-variance derivative estimates.

In [ ]:
# ============================================================
# TASK 2.3 — Estimate dV/dlambda
# ============================================================
print("=" * 60)
print("TASK 2.3 — Estimating dV/dlambda")
print("=" * 60)

# ── Hyper-parameters for the comparison ──────────────────────────
N_DIM123    = 200_000    # N fixed for dims-1/2/3 comparison
DL_DIM123   = 0.05       # Δλ fixed for dims-1/2/3 comparison
BASE_SEED   = 42
K_23        = K_obs      # K = 60 as specified

METHODS_23  = ['standard', 'antithetic', 'moment_matching', 'halton', 'sobol']
LABELS_23   = ['Standard', 'Antithetic', 'Moment\nMatching', 'Halton', 'Sobol']
APPROXES_23 = ['forward', 'central', 'backward']
APRLABELS   = ['Forward', 'Central', 'Backward']

# ================================================================
#  CORE ENGINE — CRN path simulator for ALL 5 MC methods
#
#  Simulates pay_lo (asset dynamics under λ = lam_lo) and
#  pay_hi (asset dynamics under λ = lam_hi) using the SAME
#  stream of random numbers for both, i.e. full CRN.
#
#  Shared randomness:
#    • u_sw   : regime-switching uniforms (shared per step)
#    • z      : normal draws / QMC draws  (shared per step)
#  The only difference between lo and hi paths is the switching
#  probability P(L→H) = lam * dt.
#
#  Returns (pay_lo, pay_hi), each shape (n_paths,).
# ================================================================

def simulate_derivative_crn(n_paths, K, lam_lo, lam_hi, seed, method):
    """
    CRN derivative engine.  All random draws are shared between the
    'lo' simulation (switching rate lam_lo) and the 'hi' simulation
    (switching rate lam_hi).
    """
    rng      = np.random.default_rng(seed)
    dt       = T_2 / K
    sqrt_dt  = np.sqrt(dt)
    p_LH_lo  = lam_lo * dt
    p_LH_hi  = lam_hi * dt
    p_HL     = mu_2   * dt
    disc     = np.exp(-r_2 * T_2)

    n_eff = (n_paths + 1) // 2 if method == 'antithetic' else n_paths

    # ── Pre-generate QMC z-draws (shape n_eff × K) if needed ─────
    z_pre = None
    if method == 'halton':
        qmc_seed = int(rng.integers(0, 2**31))
        hal = qmc.Halton(d=K, scramble=True, seed=qmc_seed)
        u   = np.clip(hal.random(n_eff), 1e-12, 1 - 1e-12)
        z_pre = norm.ppf(u)         # (n_eff, K)  shared for lo & hi
    elif method == 'sobol':
        qmc_seed = int(rng.integers(0, 2**31))
        m   = max(1, int(np.ceil(np.log2(max(n_eff, 2)))))
        sob = qmc.Sobol(d=K, scramble=True, seed=qmc_seed)
        u   = np.clip(sob.random(2**m)[:n_eff], 1e-12, 1 - 1e-12)
        z_pre = norm.ppf(u)         # (n_eff, K)  shared for lo & hi

    # ── Initialise prices and regime state ────────────────────────
    p_lo = np.full(n_eff, S0_2, dtype=np.float64)
    p_hi = np.full(n_eff, S0_2, dtype=np.float64)
    ih_lo = np.zeros(n_eff, dtype=bool)
    ih_hi = np.zeros(n_eff, dtype=bool)
    s_lo  = np.zeros(n_eff)
    s_hi  = np.zeros(n_eff)

    if method == 'antithetic':
        p_lo_a  = np.full(n_eff, S0_2, dtype=np.float64)
        p_hi_a  = np.full(n_eff, S0_2, dtype=np.float64)
        ih_lo_a = np.zeros(n_eff, dtype=bool)
        ih_hi_a = np.zeros(n_eff, dtype=bool)
        s_lo_a  = np.zeros(n_eff)
        s_hi_a  = np.zeros(n_eff)

    # ── Step through K time intervals ─────────────────────────────
    for k in range(K):
        # Shared switching uniforms
        u_sw   = rng.random(n_eff)
        sw_lo  = np.where(ih_lo, u_sw < p_HL, u_sw < p_LH_lo)
        sw_hi  = np.where(ih_hi, u_sw < p_HL, u_sw < p_LH_hi)
        ih_lo  = np.where(sw_lo, ~ih_lo, ih_lo)
        ih_hi  = np.where(sw_hi, ~ih_hi, ih_hi)

        vol_lo = np.where(ih_lo, sigma_H2, sigma_L2)
        vol_hi = np.where(ih_hi, sigma_H2, sigma_L2)

        # Shared z draws (or QMC draws)
        if z_pre is not None:
            z = z_pre[:, k]
        elif method == 'moment_matching':
            z_raw = rng.standard_normal(n_eff)
            s_std = z_raw.std(ddof=0)
            z     = (z_raw - z_raw.mean()) / s_std if s_std > 0 else z_raw
        else:
            z = rng.standard_normal(n_eff)

        p_lo = p_lo * np.exp((r_2 - 0.5*vol_lo**2)*dt + vol_lo*sqrt_dt* z)
        p_hi = p_hi * np.exp((r_2 - 0.5*vol_hi**2)*dt + vol_hi*sqrt_dt* z)
        s_lo += p_lo;  s_hi += p_hi

        # Antithetic mirror paths (share u_sw, use −z)
        if method == 'antithetic':
            sw_lo_a = np.where(ih_lo_a, u_sw < p_HL, u_sw < p_LH_lo)
            sw_hi_a = np.where(ih_hi_a, u_sw < p_HL, u_sw < p_LH_hi)
            ih_lo_a = np.where(sw_lo_a, ~ih_lo_a, ih_lo_a)
            ih_hi_a = np.where(sw_hi_a, ~ih_hi_a, ih_hi_a)

            vol_lo_a = np.where(ih_lo_a, sigma_H2, sigma_L2)
            vol_hi_a = np.where(ih_hi_a, sigma_H2, sigma_L2)

            p_lo_a = p_lo_a * np.exp((r_2 - 0.5*vol_lo_a**2)*dt + vol_lo_a*sqrt_dt*(-z))
            p_hi_a = p_hi_a * np.exp((r_2 - 0.5*vol_hi_a**2)*dt + vol_hi_a*sqrt_dt*(-z))
            s_lo_a += p_lo_a;  s_hi_a += p_hi_a

    pay_lo = disc * np.maximum(s_lo/K - p_lo, 0.0)
    pay_hi = disc * np.maximum(s_hi/K - p_hi, 0.0)

    if method == 'antithetic':
        pay_lo_a = disc * np.maximum(s_lo_a/K - p_lo_a, 0.0)
        pay_hi_a = disc * np.maximum(s_hi_a/K - p_hi_a, 0.0)
        pay_lo   = np.concatenate([pay_lo,   pay_lo_a])[:n_paths]
        pay_hi   = np.concatenate([pay_hi,   pay_hi_a])[:n_paths]

    return pay_lo, pay_hi


# ================================================================
#  UNIFIED DERIVATIVE ESTIMATOR
#  Returns (estimate, SE) given all 5 dimensions:
#    method   : one of METHODS_23
#    use_crn  : True = CRN, False = independent
#    approx   : 'forward' | 'backward' | 'central'
#    n_paths  : number of simulation paths
#    delta_lam: step size Δλ
# ================================================================

_SIM_FUNCS = {
    'standard':        simulate_asian_put,
    'antithetic':      simulate_asian_put_antithetic,
    'moment_matching': simulate_asian_put_moment_matching,
    'halton':          simulate_asian_put_halton,
    'sobol':           simulate_asian_put_sobol,
}

def compute_deriv_estimate(n_paths, K, lam_val, delta_lam,
                            seed, method, use_crn, approx):
    """
    Estimate dV/dλ and its standard error.

    CRN:
      The same random stream is used for both the base simulation
      and the perturbed simulation; diff = (pay_hi − pay_lo) / step
      is computed path-by-path, so covariance is exploited.

    Independent:
      Base and perturbed simulations use separate seeds; SE is
      derived from the sum of individual variances.
    """
    if use_crn:
        # Map approximation type to (lam_lo, lam_hi, denominator)
        if approx == 'forward':
            lo, hi = lam_val, lam_val + delta_lam
            denom  = delta_lam
        elif approx == 'backward':
            lo, hi = lam_val - delta_lam, lam_val
            denom  = delta_lam
        elif approx == 'central':
            lo, hi = lam_val - delta_lam, lam_val + delta_lam
            denom  = 2 * delta_lam

        pay_lo, pay_hi = simulate_derivative_crn(
            n_paths, K, lo, hi, seed, method)
        diff = (pay_hi - pay_lo) / denom
        est  = diff.mean()
        se   = diff.std(ddof=1) / np.sqrt(len(diff))

    else:
        # Independent sampling
        sim = _SIM_FUNCS[method]
        OFFSET = 7919   # large prime; ensures distinct seed for bumped run

        if approx == 'forward':
            pa = sim(n_paths, K=K, lam_val=lam_val,              seed=seed)
            pb = sim(n_paths, K=K, lam_val=lam_val + delta_lam,  seed=seed + OFFSET)
            denom = delta_lam
        elif approx == 'backward':
            pa = sim(n_paths, K=K, lam_val=lam_val - delta_lam,  seed=seed)
            pb = sim(n_paths, K=K, lam_val=lam_val,              seed=seed + OFFSET)
            denom = delta_lam
        elif approx == 'central':
            pa = sim(n_paths, K=K, lam_val=lam_val - delta_lam,  seed=seed)
            pb = sim(n_paths, K=K, lam_val=lam_val + delta_lam,  seed=seed + OFFSET)
            denom = 2 * delta_lam

        est = (pb.mean() - pa.mean()) / denom
        se  = np.sqrt(pa.var(ddof=1) + pb.var(ddof=1)) / (np.sqrt(n_paths) * denom)

    return est, se

### Step 1 — Compare Dimensions 1/2/3 (Sampling × Method × Approximation)

Evaluate all 30 combinations (2 sampling strategies × 5 MC methods × 3 finite-difference approximations) and display results as a dual heatmap.

In [ ]:
# ================================================================
#  STEP 1  — Compare dims 1, 2, 3 simultaneously
#  Fix: N = N_DIM123, Δλ = DL_DIM123
#  Metric: SE (standard deviation of the derivative estimate)
#
#  VISUALISATION: 2-panel heatmap
#    Left panel  = CRN,  Right panel = Independent
#    Rows  = MC method (5 options)
#    Cols  = Approximation type (3 options)
#    Color = SE  (lower = better)
#
#  Rationale for heatmap:
#    • 30 combinations fit cleanly without clutter
#    • Color gradient lets the eye find the global minimum instantly
#    • Cell annotations give exact SE values for the report
# ================================================================
print("=" * 70)
print("TASK 2.3 REVISED — Systematic 5-Dimension Selection for dV/dλ")
print("=" * 70)
print(f"\nStep 1: SE comparison across dims 1/2/3")
print(f"(N = {N_DIM123:,}, Δλ = {DL_DIM123}, K = {K_23})\n")

se_grid = {}   # key: (samp_label, method, approx) → SE
est_grid = {}

for samp_label, use_crn in [('crn', True), ('independent', False)]:
    for method in METHODS_23:
        for approx in APPROXES_23:
            est, se = compute_deriv_estimate(
                N_DIM123, K_23, lam_2, DL_DIM123,
                BASE_SEED, method, use_crn, approx)
            se_grid[(samp_label, method, approx)]  = se
            est_grid[(samp_label, method, approx)] = est
            print(f"  {samp_label:12s} | {method:15s} | {approx:8s} "
                  f"→ est = {est:+.5f},  SE = {se:.6f}")

# ── Build heatmap arrays ──────────────────────────────────────────
crn_data  = np.array([[se_grid[('crn',         m, a)] for a in APPROXES_23]
                       for m in METHODS_23])
ind_data  = np.array([[se_grid[('independent', m, a)] for a in APPROXES_23]
                       for m in METHODS_23])

vmin_global = min(crn_data.min(), ind_data.min())
vmax_global = max(crn_data.max(), ind_data.max())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax_i, (title_str, data) in enumerate(
        [("CRN (Common Random Numbers)", crn_data),
         ("Independent Sampling",        ind_data)]):

    im = axes[ax_i].imshow(
        data, aspect='auto', cmap='YlOrRd',
        vmin=vmin_global, vmax=vmax_global)

    axes[ax_i].set_xticks(range(len(APPROXES_23)))
    axes[ax_i].set_xticklabels(APRLABELS, fontsize=10)
    axes[ax_i].set_yticks(range(len(METHODS_23)))
    axes[ax_i].set_yticklabels(LABELS_23, fontsize=10)
    axes[ax_i].set_title(title_str, fontsize=11, fontweight='bold')
    axes[ax_i].set_xlabel("Approximation Method", fontsize=10)
    if ax_i == 0:
        axes[ax_i].set_ylabel("MC Method", fontsize=10)

    # Annotate each cell with the SE value
    for r in range(len(METHODS_23)):
        for c in range(len(APPROXES_23)):
            val        = data[r, c]
            brightness = (val - vmin_global) / (vmax_global - vmin_global + 1e-15)
            tc         = 'white' if brightness > 0.65 else 'black'
            axes[ax_i].text(c, r, f"{val:.5f}",
                            ha='center', va='center',
                            fontsize=8, color=tc, fontweight='bold')

    plt.colorbar(im, ax=axes[ax_i], label='SE', fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# ── Identify best combo (lowest SE) ─────────────────────────────
best_key   = min(se_grid, key=se_grid.get)
best_samp, best_method, best_approx = best_key
best_crn   = (best_samp == 'crn')
print(f"\n>>> BEST combo (lowest SE across all 30 combinations):")
print(f"    Sampling    : {best_samp.upper()}")
print(f"    MC Method   : {best_method}")
print(f"    Approx type : {best_approx}")
print(f"    SE          : {se_grid[best_key]:.6f}")
print(f"    Estimate    : {est_grid[best_key]:+.5f}")

### Step 2 — SE vs N (Sweep Number of Paths)

Using the best combination from Step 1, sweep N from 200K to 2M to find the optimal path count.

In [ ]:
# ================================================================
#  STEP 2  — SE vs N
#  Using best (method, sampling, approx) from Step 1,
#  sweep N to find the best number of paths.
#  Metric: SE  (lower = better).
# ================================================================
N_SWEEP = list(range(200_000, 2_000_001, 200_000))

print(f"\nStep 2: SE vs N")
print(f"(method={best_method}, sampling={best_samp}, approx={best_approx}, "
      f"Δλ={DL_DIM123})\n")

se_vs_N  = []
est_vs_N = []

for N in N_SWEEP:
    est, se = compute_deriv_estimate(
        N, K_23, lam_2, DL_DIM123,
        BASE_SEED, best_method, best_crn, best_approx)
    se_vs_N.append(se)
    est_vs_N.append(est)
    print(f"  N = {N:>9,}:  est = {est:+.5f},  SE = {se:.6f}")

# ── Plot: SE vs N ────────────────────────────────────────────────
N_arr = np.array(N_SWEEP, dtype=float)
ref_1_sqrt_N = se_vs_N[0] * np.sqrt(N_SWEEP[0]) / np.sqrt(N_arr)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_SWEEP, se_vs_N, 'o-',
        color='tab:blue', linewidth=2, markersize=7,
        label=f"{best_samp.upper()}, {best_method}, {best_approx}")

# ── Trục X ────────────────────────────────────────────────────────
ax.set_xticks(N_SWEEP)
ax.set_xticklabels([f"{n//1000}K" if n < 2_000_001 else "2M"
                    for n in N_SWEEP], fontsize=9)

# ── Trục Y: ticks đều, format số thập phân ────────────────────────
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f"{x:.4f}".rstrip('0').rstrip('.')))

ax.set_xlabel("Number of paths ($N$)", fontsize=12)
ax.set_ylabel("Standard Error of $dV/d\\lambda$ estimate", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='major', alpha=0.5, linewidth=0.8)
plt.tight_layout()
plt.show()

# ── Choose best N ────────────────────────────────────────────────
# Use the largest N that gives a good SE-vs-time tradeoff.
# Here we simply take the largest N in the sweep.
best_N = N_SWEEP[-1]
print(f"\n>>> BEST N = {best_N:,}  "
      f"(SE = {se_vs_N[N_SWEEP.index(best_N)]:.6f})")

### Step 3 — MSE vs Δλ (Bias-Variance Trade-off)

For each Δλ, run multiple replications to decompose MSE into Bias² and Variance, then find the optimal step size.

In [ ]:
# ================================================================
#  STEP 3  — MSE vs Δλ
#  Approach:
#    1. Compute a high-accuracy reference value (large N + central CRN).
#    2. For each Δλ, run R independent replications with N = N_MSE.
#    3. Bias(Δλ)     = mean(estimates) − reference
#    4. Variance(Δλ) = var(estimates over replications)
#    5. MSE(Δλ)      = Bias² + Variance
#
#  Numerical insight:
#    • Small Δλ  → Bias² ↓  but Variance ↑  (cancellation noise dominates)
#    • Large Δλ  → Variance ↓  but Bias² ↑  (truncation error dominates)
#    • Optimal Δλ minimises MSE (the two effects balance).
# ================================================================
print(f"\nComputing high-accuracy reference for dV/dλ …")
ref_est, ref_se = compute_deriv_estimate(
    2_000_000, K_23, lam_2, 0.02,
    BASE_SEED, 'antithetic', True, 'central')
print(f"  Reference estimate : {ref_est:+.6f}  (SE = {ref_se:.7f})")

DL_VALUES = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]
N_REPS    = 20          # replications per Δλ to estimate bias & variance
N_MSE_REP = 200_000     # N per replication

print(f"\nStep 3: MSE vs Δλ")
print(f"(R = {N_REPS} reps × N = {N_MSE_REP:,} per rep, "
      f"method={best_method}, sampling={best_samp}, approx={best_approx})\n")

mse_vals  = []
bias2_vals = []
var_vals  = []
mean_vals = []

print(f"{'Δλ':>8}  {'mean_est':>10}  {'Bias':>10}  "
      f"{'Bias²':>12}  {'Variance':>12}  {'MSE':>12}")
print("-" * 72)

for dl in DL_VALUES:
    rep_ests = []
    for rep in range(N_REPS):
        est_r, _ = compute_deriv_estimate(
            N_MSE_REP, K_23, lam_2, dl,
            BASE_SEED + rep * 137,           # distinct seed per replication
            best_method, best_crn, best_approx)
        rep_ests.append(est_r)

    rep_ests  = np.array(rep_ests)
    mean_est  = rep_ests.mean()
    bias      = mean_est - ref_est
    variance  = rep_ests.var(ddof=1)
    mse       = bias**2 + variance

    mse_vals.append(mse);  bias2_vals.append(bias**2)
    var_vals.append(variance);  mean_vals.append(mean_est)

    print(f"{dl:8.4f}  {mean_est:+10.5f}  {bias:+10.5f}  "
          f"{bias**2:12.6f}  {variance:12.6f}  {mse:12.6f}")

# ── Plot: MSE vs Δλ ──────────────────────────────────────────────
best_dl_idx = int(np.argmin(mse_vals))
best_dl     = DL_VALUES[best_dl_idx]

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(DL_VALUES, mse_vals,   'k-o',  linewidth=2,   markersize=7, label='MSE = Bias² + Var',  zorder=3)
ax.loglog(DL_VALUES, bias2_vals, 'r--s', linewidth=1.8, markersize=6, label='Bias²',               zorder=2)
ax.loglog(DL_VALUES, var_vals,   'b:^',  linewidth=1.8, markersize=6, label='Variance',            zorder=2)
ax.axvline(best_dl, color='green', linestyle='-.',
           linewidth=1.8, label=f'Optimal $\\Delta\\lambda$ = {best_dl}', zorder=4)

# ── Trục X: hiển thị đúng các giá trị Δλ đã dùng ─────────────────
ax.set_xticks(DL_VALUES)
ax.set_xticklabels([str(dl) for dl in DL_VALUES], rotation=45, fontsize=8.5)

# ── Trục Y: ticks dày hơn trải đều trên log scale ─────────────────
ax.yaxis.set_major_locator(mticker.LogLocator(base=10, numticks=15))
ax.yaxis.set_minor_locator(mticker.LogLocator(base=10,
                            subs=[2,3,4,5,6,7,8,9], numticks=50))
ax.yaxis.set_major_formatter(mticker.LogFormatterSciNotation(base=10))
ax.yaxis.set_minor_formatter(mticker.NullFormatter())

ax.set_xlabel("$\\Delta\\lambda$", fontsize=13)
ax.set_ylabel("MSE / Bias² / Variance", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='major', alpha=0.5, linewidth=0.8)
ax.grid(True, which='minor', alpha=0.25, linewidth=0.4, linestyle=':')
plt.tight_layout()
plt.show()

### Final Answer — Best Settings Across All 5 Dimensions

Combine the best sampling strategy, MC method, approximation type, N, and Δλ to produce the final dV/dλ estimate.

In [ ]:
# ================================================================
#  FINAL ANSWER
#  Combine best settings from all 5 dimensions and report.
# ================================================================
print("\n" + "=" * 70)
print("FINAL ESTIMATE — Best settings from all 5 dimensions")
print("=" * 70)
print(f"  Dim 1 — Sampling  : {best_samp.upper()}")
print(f"  Dim 2 — MC Method : {best_method}")
print(f"  Dim 3 — Approx    : {best_approx}")
print(f"  Dim 4 — N         : {best_N:,}")
print(f"  Dim 5 — Δλ        : {best_dl}")
print()

final_est, final_se = compute_deriv_estimate(
    best_N, K_23, lam_2, best_dl,
    BASE_SEED, best_method, best_crn, best_approx)

print(f"  dV/dλ estimate : {final_est:+.6f}")
print(f"  Standard Error : {final_se:.7f}")
print(f"  95% CI         : [{final_est - 1.96*final_se:.6f}, "
      f"{final_est + 1.96*final_se:.6f}]")